## Strain profiles

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from skimage.measure import profile_line
from scipy import ndimage
from scipy.ndimage import gaussian_filter1d
#%matplotlib inline

In [ ]:
def ice_velocity(u_0, y, w, n, option=0):
    '''
    Calculate the transect velocity knowing the centrale velocity of the profile 
    u_0    --> velocity at the centrer of the profile (at y = 0) 
    y      --> Y vector position along a profile perpendicular to the flow, y = 0 at the center
    w      --> half-width of the profile
    n      --> stress exponent (see Glen's flow law)
    option --> 0 for y = 0 a center, 1 for y = 0 at margin
    '''
    if option == 0:
        u = (1 - (abs(y)/w) ** (n+1)) * u_0
    elif option == 1:
        u = (1 - (abs(1 - y/w)) ** (n+1)) * u_0
    return u

def pad_center(arr, target_len):
    pad_total = target_len - len(arr)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return np.pad(arr, (pad_left, pad_right), constant_values=np.nan)

def L2_norm_residuals(data, model, variance):
    '''
    Calculate L2-norm (Euclidean distance) between data and model
    '''
    # Calculate the misfit with weights (variance)
    weighted_residuals = (data - model) ** 2 / variance
    misfit = np.sqrt(np.nansum(weighted_residuals) / len(data))
    return misfit

### Load data

In [ ]:
path = 'velocity_data/'
years_list = [2015, 2016, 2017, 2018]

img15 = rasterio.open(path + 'median_2015_cleaned2.tif')
img16 = rasterio.open(path + 'median_2016_cleaned2.tif')
img17 = rasterio.open(path + 'median_2017.tif')
img18 = rasterio.open(path + 'median_2018.tif')

# Define the profile coordinates:
x0, y0 = 650, 435
x1, y1 = 690, 670

# Medin-filter images to remove edge peaks
img15_filt = ndimage.median_filter(img15.read(1), 20)
img16_filt = ndimage.median_filter(img16.read(1), 20)
img17_filt = ndimage.median_filter(img17.read(1), 20)
img18_filt = ndimage.median_filter(img18.read(1), 20)

# Velocity profiles of the different years
u15 = profile_line(img15_filt, (y0, x0), (y1, x1))
u16 = profile_line(img16_filt, (y0, x0), (y1, x1))
u17 = profile_line(img17_filt, (y0, x0), (y1, x1))
u18 = profile_line(img18_filt, (y0, x0), (y1, x1))

# Find U_0 for each year
u0_15, idx15 = np.max(u15), np.argmax(u15)
u0_16, idx16 = np.max(u16), np.argmax(u16)
u0_17, idx17 = np.max(u17), np.argmax(u17)
u0_18, idx18 = np.max(u18), np.argmax(u18)

# Original y vector
y = np.linspace(-len(u15)/2*30, len(u15)/2*30-30, len(u15))


### Estimate strain on velocity data

In [ ]:
# Calcul of strain:
# 0 - Filter velocity data
sigma = 3 # filter parameter
u_filtered = [gaussian_filter1d(u15, sigma),
              gaussian_filter1d(u16, sigma),
              gaussian_filter1d(u17, sigma),
              gaussian_filter1d(u18, sigma)]

#u_filtered = [u15, u16, u17, u18]

# 1 - Instantaneous strain: dU/dy (in day^-1) x 365 (to convert to yr^-1) x dt (total strain of the year, dt = 1yr)
strain = [np.gradient(u_filtered[0],y) * 365 * 1,
          np.gradient(u_filtered[1],y) * 365 * 1,
          np.gradient(u_filtered[2],y) * 365 * 1,
          np.gradient(u_filtered[3],y) * 365 * 1]


# 2 - Total strain = cumulative strain over the years:
strain_total = [strain[0],
                strain[0]+strain[1],
                strain[0]+strain[1]+strain[2],
                strain[0]+strain[1]+strain[2]+strain[3]]

plt.figure(figsize=(12,8))
plt.ylabel('Surface velocity (m/day)')
plt.subplot(221)
plt.plot(y,u15, 'blue', linestyle='--')
plt.plot(y,u16, 'orange', linestyle='--')
plt.plot(y,u17, 'green', linestyle='--')
plt.plot(y,u18, 'C3', linestyle='--')
plt.plot(y,u_filtered[0], 'blue')
plt.plot(y,u_filtered[1], 'orange')
plt.plot(y,u_filtered[2], 'green')
plt.plot(y,u_filtered[3], 'C3')
plt.ylabel('Smoothed surface velocity (m/day)')
plt.subplot(223)
plt.plot(y,abs(strain[0]), 'blue')
plt.plot(y,abs(strain[1]), 'orange')
plt.plot(y,abs(strain[2]), 'green')
plt.plot(y,abs(strain[3]), 'C3')
plt.ylabel('Instantaneous strain')
plt.subplot(222)
plt.plot(y,abs(strain_total[0]), 'blue')
plt.plot(y,abs(strain_total[1]), 'orange')
plt.plot(y,abs(strain_total[2]), 'green')
plt.plot(y,abs(strain_total[3]), 'C3')
plt.ylabel('Total strain')
plt.subplot(224)
plt.plot(y,abs(strain_total[0]), 'blue')
plt.plot(y,abs(strain_total[1]), 'orange')
plt.plot(y,abs(strain_total[2]), 'green')
plt.plot(y,abs(strain_total[3]), 'C3')
plt.axhline(2, color="k", linestyle="--")
plt.ylim([0,5])
plt.ylabel('Total strain')
plt.legend([2015,2016,2017,2018])

In [ ]:
threshold = 2
years = [2015, 2016, 2017, 2018]
crossings_interp = []
for i in range(4):
    strain = np.abs(strain_total[i])                                    # Get absolute total strain
    y_cross = []
    for j in range(len(y)-1):                                           # Loop over y axis samples
        if (strain[j] - threshold) * (strain[j+1] - threshold) < 0:     # Test if current sample and next one cross strain > 2 threshold
            y0, y1 = y[j], y[j+1]
            s0, s1 = strain[j], strain[j+1]
            y_interp = y0 + (threshold - s0) * (y1 - y0) / (s1 - s0)    # Linear interpolation to get more accurate location
            y_cross.append(y_interp)                                    # Save y boundary
    crossings_interp.append(y_cross)                                    # Save y boundaries for this year
    print(f'Year {years[i]} crossings (interp): {y_cross}')

y_left_limit  = [crossings_interp[0][1],
                 crossings_interp[1][1],
                 crossings_interp[2][1],
                 crossings_interp[3][1]]     # 2nd crossing
y_right_limit = [crossings_interp[0][-2],
                 crossings_interp[1][-2],
                 crossings_interp[2][-2],
                 crossings_interp[3][-2]]    # 3rd crossing (one before last)


## Compute separated models for center and edge part of 2016

In [ ]:
# Choose velocity year to study and parameters
year = '2016'                                   # select year
U_VEC = u16                                     # select the velocity profile
ERR = np.ones((len(U_VEC))) #err15              # select error vector (variance profiles). To inverse without error, put ERR = np.ones((len(U_VEC)))
Y_VEC = y                                       # select the y vector

# Create center and edge model
y_left_limit_2016 = y_left_limit[1]
y_right_limit_2016 = y_right_limit[1]
mask_edges = (Y_VEC <= y_left_limit_2016) | (Y_VEC >= y_right_limit_2016)
mask_center = (Y_VEC > y_left_limit_2016) & (Y_VEC < y_right_limit_2016)
Y_edges = Y_VEC[mask_edges]
U_edges = U_VEC[mask_edges]
U0_center = np.max(U_VEC[mask_center])
U0_edges  = np.max(U_VEC[mask_edges])

# Run the inversion
n_list = np.linspace(0,12,61).round(2)                                              # list of stress exponent
w_list = np.linspace(2100,3900,61)                                                  # list of half-width  85*30 - 145*30
RES_center = np.empty((len(w_list), len(n_list)))
RES_edges  = np.empty((len(w_list), len(n_list)))

for i, w in enumerate(w_list):

    # --- restrict full profile to current width ---
    mask_w = np.abs(Y_VEC) <= w
    
    Y_sub = Y_VEC[mask_w]
    U_sub = U_VEC[mask_w]

    # corresponding masks
    mask_center_sub = mask_center[mask_w]
    mask_edges_sub  = mask_edges[mask_w]

    # define model grid (same as data)
    Y_model = np.linspace(-w, w, len(Y_sub))

    for j, n in enumerate(n_list):

        # --- build BOTH models on full domain ---
        U_model_center = ice_velocity(U0_center, Y_model, w, n)
        U_model_edges  = ice_velocity(U0_edges,  Y_model, w, n)

        # --- compute residuals ONLY on relevant regions ---
        if np.sum(mask_center_sub) > 5:
            RES_center[i,j] = L2_norm_residuals(
                U_sub[mask_center_sub],
                U_model_center[mask_center_sub],
                np.ones(np.sum(mask_center_sub))
            )
        else:
            RES_center[i,j] = np.nan

        if np.sum(mask_edges_sub) > 5:
            RES_edges[i,j] = L2_norm_residuals(
                U_sub[mask_edges_sub],
                U_model_edges[mask_edges_sub],
                np.ones(np.sum(mask_edges_sub))
            )
        else:
            RES_edges[i,j] = np.nan

In [ ]:

# -------------------------------- CENTER ----------------------------- #
# Find best model
min_misfit_center = np.min(RES_center)
idx_min_center = np.argwhere(RES_center == min_misfit_center)
best_w_center = w_list[idx_min_center[0,0]]
best_n_center = n_list[idx_min_center[0,1]]

threshold = min_misfit_center * 1.05 # Define threshold (5% above min)
N, W = np.meshgrid(n_list, w_list)

misfit = RES_center
valid_w_n_indices = np.where(misfit <= threshold)
valid_n_indices = np.where(misfit <= threshold)[1]
valid_n_indices.sort()
print(f'5% best models for year {year} are n between {n_list[valid_n_indices[0]]} and {n_list[valid_n_indices[-1]]} (best n = {best_n_center})')

# --- Plot residuals ---
plt.figure(figsize=(20,5))
plt.subplot(131)
im = plt.imshow(RES_center, cmap='RdBu_r', origin='lower', extent=[n_list.min(), n_list.max(), w_list.min(), w_list.max()], aspect='auto')
CS = plt.contour(N, W, RES_center, levels=[threshold], colors='yellow', linewidths=2)
CS.collections[0].set_label('5% threshold')
plt.scatter(best_n_center, best_w_center, s=120, color='yellow', edgecolor='black', zorder=5, label=f'Best model: n={round(best_n_center,1)}, w={round(best_w_center)}')
plt.xlabel('Stress exponent n')
plt.ylabel('Half-width w')
plt.title('Central profile')
plt.legend()
plt.colorbar(im, label='RMS misfit')

# -------------------------------- EDGES ----------------------------- #
# Find best model
min_misfit_edges = np.min(RES_edges)
idx_min_edges = np.argwhere(RES_edges == min_misfit_edges)
best_w_edges = w_list[idx_min_edges[0,0]]
best_n_edges = n_list[idx_min_edges[0,1]]

threshold = min_misfit_edges * 1.05 # Define threshold (5% above min)
N, W = np.meshgrid(n_list, w_list)

misfit = RES_edges
valid_w_n_indices = np.where(misfit <= threshold)
valid_n_indices = np.where(misfit <= threshold)[1]
valid_n_indices.sort()
print(f'5% best models for year {year} are n between {n_list[valid_n_indices[0]]} and {n_list[valid_n_indices[-1]]} (best n = {best_n_edges})')

# --- Plot residuals ---
plt.subplot(133)
im = plt.imshow(RES_edges, cmap='RdBu_r', origin='lower', extent=[n_list.min(), n_list.max(), w_list.min(), w_list.max()], aspect='auto')
CS = plt.contour(N, W, RES_edges, levels=[threshold], colors='yellow', linewidths=2)
CS.collections[0].set_label('5% threshold')
plt.scatter(best_n_edges, best_w_edges, s=120, color='yellow', edgecolor='black', zorder=5, label=f'Best model: n={round(best_n_edges,1)}, w={round(best_w_edges)}')
plt.xlabel('Stress exponent n')
plt.ylabel('Half-width w')
plt.title('Edges profile')
plt.legend()
plt.colorbar(im, label='RMS misfit')

# -------------------------------- MODELS ----------------------------- #

mask_w_center = np.abs(Y_VEC) <= best_w_center
Y_DATA = Y_VEC[mask_w_center]
U_DATA_center = U_VEC[np.where((Y_VEC > -best_w_center) & (Y_VEC < best_w_center))]      # limit the real U profile to the same as modeled
Y_MODEL_center = np.linspace(-best_w_center,best_w_center,int(len(Y_DATA)))              # estimation of the y vector for modeled U profile
U_BEST_MODEL_center = ice_velocity(U0_center, Y_MODEL_center, best_w_center, best_n_center)           # calcul of the modeled U

mask_w_edges = np.abs(Y_VEC) <= best_w_edges
Y_DATA = Y_VEC[mask_w_edges]
U_DATA_edges = U_VEC[np.where((Y_VEC > -best_w_edges) & (Y_VEC < best_w_edges))]      # limit the real U profile to the same as modeled
Y_MODEL_edges = np.linspace(-best_w_edges,best_w_edges,int(len(Y_DATA)))              # estimation of the y vector for modeled U profile
U_BEST_MODEL_edges = ice_velocity(U0_edges, Y_MODEL_edges, best_w_edges, best_n_edges)           # calcul of the modeled U

# Plot best model
plt.subplot(132)
plt.axvline(y_left_limit_2016, color='grey', linestyle='--')
plt.axvline(y_right_limit_2016, color='grey', linestyle='--')
plt.plot(Y_VEC,U_VEC/U0_center, color='k', linewidth=3, label='Raw velocity')
plt.plot(Y_MODEL_center,U_BEST_MODEL_center/U0_center, color='red', linewidth=3, label='Modeled velocity center')
plt.plot(Y_MODEL_edges,U_BEST_MODEL_edges/U0_center, color='orange', linewidth=3, label='Modeled velocity edges')

plt.legend()
plt.xlabel('y position (m)')
plt.ylabel('u(y) / u(y=0)')